# Bronze Layer — Data Ingestion Overview

Surveys every table ingested into the **bronze_lakehouse** so you can confirm what landed and inspect it.

| | |
|---|---|
| **Lakehouse** | bronze_lakehouse |
| **AOI** | Maple Ridge, BC — 20 km radius |
| **Layers** | Planetary Computer STAC, PC collections, DataBC WFS geology |

Run top-to-bottom. The default lakehouse must be attached (`bronze_lakehouse`).

## 1. Discover all bronze tables

In [ ]:
# List every table in the default lakehouse and keep the bronze_ ones
all_tables = [t.name for t in spark.catalog.listTables()]
bronze_tables = sorted([t for t in all_tables if t.startswith("bronze_")])

print(f"Found {len(bronze_tables)} bronze tables:\n")
for t in bronze_tables:
    print(f"  - {t}")

## 2. Row counts per table

In [ ]:
# Build a summary of row counts and column counts for each bronze table
from pyspark.sql import Row

summary = []
for t in bronze_tables:
    try:
        df = spark.table(t)
        summary.append(Row(table=t, rows=df.count(), columns=len(df.columns)))
    except Exception as e:
        summary.append(Row(table=t, rows=-1, columns=-1))
        print(f"  ! {t}: {e}")

summary_df = spark.createDataFrame(summary).orderBy("table")
print(f"Total bronze tables: {len(summary)}")
print(f"Total rows across all bronze tables: {sum(r.rows for r in summary if r.rows > 0)}")
display(summary_df)

## 3. Schema + sample rows for each table

In [ ]:
# Inspect schema and a few sample rows for every bronze table
for t in bronze_tables:
    print("=" * 80)
    print(f"TABLE: {t}")
    print("=" * 80)
    try:
        df = spark.table(t)
        print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
        df.printSchema()
        display(df.limit(5))
    except Exception as e:
        print(f"  ! Could not read {t}: {e}")
    print()

## 4. Map — spatial footprint of every bronze layer

Interactive map of all bronze data over the Maple Ridge AOI:

- **STAC tables** (satellite / elevation / land cover) → drawn as item **bounding boxes**.
- **DataBC geology tables** (quaternary, bedrock, faults) → drawn as their actual **GeoJSON geometries**.
- Red marker + circle = AOI center and 20 km radius.

Each table is a toggleable layer (use the layer control, top-right).

In [ ]:
# Interactive map of every bronze layer's spatial footprint
import sys, subprocess, json

try:
    import folium
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"], check=True)
    import folium

LAT, LON, RADIUS_KM = 49.2193, -122.5984, 20

m = folium.Map(location=[LAT, LON], zoom_start=10, tiles="CartoDB positron")

# AOI center + radius
folium.Marker(
    [LAT, LON], tooltip="AOI center — Maple Ridge, BC",
    icon=folium.Icon(color="red", icon="star"),
).add_to(m)
folium.Circle(
    [LAT, LON], radius=RADIUS_KM * 1000, color="red",
    fill=False, weight=2, tooltip=f"{RADIUS_KM} km AOI",
).add_to(m)

# Distinct CSS-safe color per table (valid for Leaflet paths)
palette = [
    "blue", "green", "purple", "orange", "darkred", "teal", "darkgreen",
    "navy", "magenta", "brown", "gray", "steelblue", "crimson", "olive",
]
color_for = {t: palette[i % len(palette)] for i, t in enumerate(bronze_tables)}

bbox_cols = {"bbox_minx", "bbox_miny", "bbox_maxx", "bbox_maxy"}
plotted = []

for t in bronze_tables:
    color = color_for[t]
    try:
        df = spark.table(t)
    except Exception as e:
        print(f"  ! skip {t}: {e}")
        continue
    cols = set(df.columns)
    fg = folium.FeatureGroup(name=t, show=True)
    n = 0

    if bbox_cols.issubset(cols):
        # STAC-style: draw item bounding-box rectangles
        rows = df.select("bbox_minx", "bbox_miny", "bbox_maxx", "bbox_maxy").limit(100).collect()
        for r in rows:
            if None in (r["bbox_minx"], r["bbox_miny"], r["bbox_maxx"], r["bbox_maxy"]):
                continue
            folium.Rectangle(
                bounds=[[r["bbox_miny"], r["bbox_minx"]], [r["bbox_maxy"], r["bbox_maxx"]]],
                color=color, weight=1, fill=True, fill_opacity=0.04, tooltip=t,
            ).add_to(fg)
            n += 1
    elif "geometry_json" in cols:
        # Geology-style: draw actual GeoJSON geometries
        rows = df.select("geometry_json").limit(1000).collect()
        for r in rows:
            gj = r["geometry_json"]
            if not gj:
                continue
            try:
                geom = json.loads(gj)
            except Exception:
                continue
            folium.GeoJson(
                geom,
                style_function=lambda x, c=color: {
                    "color": c, "weight": 2, "fillColor": c, "fillOpacity": 0.2,
                },
                tooltip=t,
            ).add_to(fg)
            n += 1
    else:
        print(f"  - {t}: no spatial columns (bbox_* or geometry_json) — not mapped")
        continue

    fg.add_to(m)
    plotted.append((t, n, color))
    print(f"  + {t}: {n} features ({color})")

folium.LayerControl(collapsed=False).add_to(m)
print(f"\nMapped {len(plotted)} bronze tables.")

# Render in Fabric
try:
    displayHTML(m._repr_html_())
except NameError:
    m